# Práctica 9. Estados de varios qubits, entrelazamiento y Bell

**Asignatura:** Computación cuántica  
**Sesión:** 9     
**Entorno recomendado:** Jupyter Notebook

En esta práctica se trabajará con sistemas de dos qubits, producto tensorial, puerta CNOT, estados de Bell, correlaciones cuánticas y una primera aproximación a la desigualdad CHSH.

## Objetivos

Al finalizar la práctica deberás ser capaz de:

- Representar estados de dos qubits mediante vectores complejos.
- Construir estados compuestos usando producto tensorial.
- Calcular probabilidades de medida para sistemas de dos qubits.
- Implementar circuitos de dos qubits en Qiskit.
- Aplicar e interpretar la puerta CNOT.
- Construir los cuatro estados de Bell.
- Medir estados entrelazados con distintos números de shots.
- Analizar correlaciones cuánticas.
- Distinguir un estado entrelazado de una mezcla clásica.
- Interpretar de forma básica el significado de CHSH.


## 1. Preparación del entorno

Ejecuta esta celda antes de comenzar. Si alguna importación falla, revisa que el entorno tenga instalados `qiskit` y `qiskit-aer`.


In [3]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

SEMILLA = 20
simulador = AerSimulator(seed_simulator=SEMILLA)


## 2. Parte A. Estados de dos qubits con NumPy

### Ejercicio 1. Base computacional de dos qubits

Define con NumPy los estados:

$$
|0\rangle,\quad |1\rangle
$$

y construye mediante `np.kron`:

$$
|00\rangle,\quad |01\rangle,\quad |10\rangle,\quad |11\rangle
$$

Muestra cada vector y comprueba su dimensión.


In [4]:
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

ket_00 = np.kron(ket_0, ket_0)
ket_01 = np.kron(ket_0, ket_1)
ket_10 = np.kron(ket_1, ket_0)
ket_11 = np.kron(ket_1, ket_1)

estados_base = {
    "|00>": ket_00,
    "|01>": ket_01,
    "|10>": ket_10,
    "|11>": ket_11,
}

for nombre, estado in estados_base.items():
    print(nombre)
    print(estado)
    print("Dimensión:", estado.shape)
    print()

|00>
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
Dimensión: (4,)

|01>
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]
Dimensión: (4,)

|10>
[0.+0.j 0.+0.j 1.+0.j 0.+0.j]
Dimensión: (4,)

|11>
[0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Dimensión: (4,)



**Responde:**

1. ¿Cuántas componentes tiene cada vector de dos qubits?
2. ¿Qué representa cada componente?
3. ¿Por qué usamos el producto tensorial y no una suma ordinaria?

**Respuesta:**

Escribe aquí tu respuesta.


### Ejercicio 2. Estado general de dos qubits

Considera el estado:

$$
|\psi\rangle =
\frac{1}{2}|00\rangle+
\frac{1}{2}|01\rangle+
\frac{1}{2}|10\rangle+
\frac{1}{2}|11\rangle
$$

Represéntalo con NumPy y comprueba su norma.

Después implementa una función que calcule las probabilidades de medir `00`, `01`, `10` y `11`.


In [5]:
def probabilidades_dos_qubits(estado):
    '''
    Calcula las probabilidades de medir 00, 01, 10 y 11.
    '''
    estado = np.asarray(estado, dtype=complex)

    if estado.shape != (4,):
        raise ValueError("Un estado de dos qubits debe tener cuatro amplitudes")

    if not np.isclose(np.linalg.norm(estado), 1):
        raise ValueError("El estado debe estar normalizado")

    probs = np.abs(estado) ** 2

    return {
        "00": float(probs[0]),
        "01": float(probs[1]),
        "10": float(probs[2]),
        "11": float(probs[3]),
    }


psi_uniforme = (ket_00 + ket_01 + ket_10 + ket_11) / 2

print("Estado:")
print(psi_uniforme)
print("Norma:", np.linalg.norm(psi_uniforme))
print("Probabilidades:")
print(probabilidades_dos_qubits(psi_uniforme))



Estado:
[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
Norma: 1.0
Probabilidades:
{'00': 0.25, '01': 0.25, '10': 0.25, '11': 0.25}


**Responde:**

1. ¿Qué probabilidad tiene cada resultado?
2. ¿Cuánto suma el total de probabilidades?
3. ¿Qué resultado se obtiene en una única medida?

**Respuesta:**

Escribe aquí tu respuesta.


### Ejercicio 3. Producto tensorial con superposiciones

Construye los estados:

$$
|+\rangle \otimes |0\rangle
$$

$$
|+\rangle \otimes |+\rangle
$$

donde:

$$
|+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}
$$

Para cada estado, muestra el vector, calcula la norma, calcula las probabilidades e indica qué resultados pueden aparecer al medir.


In [ ]:
ket_plus = (ket_0 + ket_1) / np.sqrt(2)

plus_0 = np.kron(ket_plus, ket_0)
plus_plus = np.kron(ket_plus, ket_plus)

for nombre, estado in {
    "|+>|0>": plus_0,
    "|+>|+>": plus_plus,
}.items():
    print(nombre)
    print(estado)
    print("Norma:", np.linalg.norm(estado))
    print("Probabilidades:", probabilidades_dos_qubits(estado))
    print()


**Responde:**

1. ¿Qué resultados pueden aparecer en $|+\rangle \otimes |0\rangle$?
2. ¿Qué resultados pueden aparecer en $|+\rangle \otimes |+\rangle$?
3. ¿Es lo mismo tener una superposición de dos qubits que tener entrelazamiento?

**Respuesta:**

Escribe aquí tu respuesta.


## 3. Parte B. Puerta CNOT

### Ejercicio 4. CNOT sobre estados de la base

Crea cuatro circuitos en Qiskit para comprobar la acción de CNOT sobre:

$$
|00\rangle,
|01\rangle,
|10\rangle,
|11\rangle
$$

Usa `Statevector.from_instruction()` para obtener el estado final.

Recuerda que en Qiskit puedes preparar estados de la base aplicando puertas `x()` a los qubits correspondientes.


In [ ]:
def preparar_estado_base(etiqueta):
    qc = QuantumCircuit(2)

    # TODO: preparar |00>, |01>, |10> o |11>
    # según el valor de etiqueta.

    return qc


def estado_final_cnot(etiqueta):
    qc = preparar_estado_base(etiqueta)
    qc.cx(0, 1)
    estado = Statevector.from_instruction(qc)
    return qc, estado


for etiqueta in ["00", "01", "10", "11"]:
    qc, estado = estado_final_cnot(etiqueta)
    print("Estado inicial:", etiqueta)
    print(estado)
    print()


Completa la tabla:

| Estado inicial | Estado final esperado | Estado obtenido |
|---|---|---|
| $\|00\rangle$ | $\|00\rangle$ |  |
| $\|01\rangle$ | $\|01\rangle$ |  |
| $\|10\rangle$ | $\|11\rangle$ |  |
| $\|11\rangle$ | $\|10\rangle$ |  |

**Responde:**

1. ¿Qué qubit actúa como control?
2. ¿Qué qubit actúa como objetivo?
3. ¿Cuándo cambia el qubit objetivo?
4. ¿Es reversible la puerta CNOT?

**Respuesta:**

Escribe aquí tu respuesta.


### Ejercicio 5. CNOT sobre una superposición

Construye el circuito:

```python
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
```

Obtén el vector de estado final y comprueba que corresponde a:

$$
|\Phi^+\rangle =
\frac{|00\rangle+|11\rangle}{\sqrt{2}}
$$


In [ ]:
qc_phi_plus = QuantumCircuit(2)
# TODO: aplicar H al qubit 0
# TODO: aplicar CNOT con control 0 y objetivo 1

estado_phi_plus = # TODO

print(estado_phi_plus)
qc_phi_plus.draw("mpl")


**Responde:**

1. ¿Qué estado hay después de aplicar $H$ al primer qubit?
2. ¿Qué efecto tiene CNOT sobre cada rama de la superposición?
3. ¿Por qué el estado final no puede interpretarse como dos qubits independientes?

**Respuesta:**

Escribe aquí tu respuesta.


## 4. Parte C. Estados de Bell

### Ejercicio 6. Construcción de los cuatro estados de Bell

Construye circuitos para preparar los cuatro estados:

$$
|\Phi^+\rangle =
\frac{|00\rangle+|11\rangle}{\sqrt{2}}
$$

$$
|\Phi^-\rangle =
\frac{|00\rangle-|11\rangle}{\sqrt{2}}
$$

$$
|\Psi^+\rangle =
\frac{|01\rangle+|10\rangle}{\sqrt{2}}
$$

$$
|\Psi^-\rangle =
\frac{|01\rangle-|10\rangle}{\sqrt{2}}
$$

Puedes partir del circuito base `H` + `CX` y añadir puertas `X` o `Z` cuando sea necesario.


In [ ]:
def circuito_phi_plus():
    qc = QuantumCircuit(2)
    # TODO
    return qc


def circuito_phi_minus():
    qc = QuantumCircuit(2)
    # TODO
    return qc


def circuito_psi_plus():
    qc = QuantumCircuit(2)
    # TODO
    return qc


def circuito_psi_minus():
    qc = QuantumCircuit(2)
    # TODO
    return qc


circuitos_bell = {
    "Phi+": circuito_phi_plus(),
    "Phi-": circuito_phi_minus(),
    "Psi+": circuito_psi_plus(),
    "Psi-": circuito_psi_minus(),
}

for nombre, qc in circuitos_bell.items():
    print(nombre)
    print(Statevector.from_instruction(qc))
    print(probabilidades_dos_qubits(Statevector.from_instruction(qc).data))
    print()


Completa:

| Estado de Bell | Amplitudes no nulas | Resultados posibles | Probabilidades |
|---|---|---|---|
| $\|\Phi^+\rangle$ |  |  |  |
| $\|\Phi^-\rangle$ |  |  |  |
| $\|\Psi^+\rangle$ |  |  |  |
| $\|\Psi^-\rangle$ |  |  |  |


### Ejercicio 7. Medida con distintos números de shots

Mide cada estado de Bell con:

```python
shots = [100, 1000, 10000]
```

Para cada caso, guarda los conteos obtenidos.


In [ ]:
def ejecutar_con_medida(qc_sin_medida, shots=1000, semilla=SEMILLA):
    """
    Recibe un circuito sin medidas, añade medidas a todos los qubits
    y devuelve los conteos de la simulación.
    """
    # TODO
    pass


shots_lista = [100, 1000, 10000]
conteos_bell = {}

for nombre, qc in circuitos_bell.items():
    conteos_bell[nombre] = {}
    for shots in shots_lista:
        conteos = # TODO
        conteos_bell[nombre][shots] = conteos
        print(nombre, "shots=", shots, conteos)
    print()


In [ ]:
# Representa un histograma para cada estado de Bell usando 1000 shots.

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes = axes.flatten()

for ax, (nombre, qc) in zip(axes, circuitos_bell.items()):
    conteos = # TODO
    plot_histogram(conteos, ax=ax, title=nombre)

plt.tight_layout()
plt.show()


**Responde:**

1. ¿Los conteos coinciden exactamente con las probabilidades teóricas?
2. ¿Qué ocurre al aumentar el número de shots?
3. ¿Qué resultados aparecen para $|\Phi^+\rangle$ y $|\Phi^-\rangle$?
4. ¿Qué resultados aparecen para $|\Psi^+\rangle$ y $|\Psi^-\rangle$?
5. ¿Permite esta medida distinguir siempre el signo $+$ del signo $-$?

**Respuesta:**

Escribe aquí tu respuesta.


## 5. Parte D. Correlaciones

### Ejercicio 8. Análisis de correlaciones

A partir de los conteos de los estados de Bell, calcula:

- Número de veces que los bits son iguales.
- Número de veces que los bits son diferentes.
- Frecuencia de bits iguales.
- Frecuencia de bits diferentes.


In [ ]:
def analizar_correlaciones(conteos):
    """
    Recibe un diccionario de conteos y calcula
    la frecuencia de resultados iguales y diferentes.
    """
    # TODO
    pass


for nombre in circuitos_bell:
    conteos = conteos_bell[nombre][1000]
    print(nombre)
    print(analizar_correlaciones(conteos))
    print()


Completa:

| Estado | Frecuencia bits iguales | Frecuencia bits diferentes | Tipo de correlación |
|---|---:|---:|---|
| $\|\Phi^+\rangle$ |  |  |  |
| $\|\Phi^-\rangle$ |  |  |  |
| $\|\Psi^+\rangle$ |  |  |  |
| $\|\Psi^-\rangle$ |  |  |  |

**Responde:**

1. ¿Qué estados producen bits iguales?
2. ¿Qué estados producen bits diferentes?
3. ¿Cada qubit individual parece aleatorio?
4. ¿Los resultados conjuntos son independientes?

**Respuesta:**

Escribe aquí tu respuesta.


### Ejercicio 9. Cada qubit individual

A partir de los conteos de $|\Phi^+\rangle$, calcula la frecuencia de 0 y 1 para cada qubit por separado.

Ten cuidado con el orden de bits mostrado por Qiskit y explica la convención que estás utilizando.


In [ ]:
def frecuencias_por_qubit(conteos):
    """
    Calcula la frecuencia de 0 y 1 para cada qubit a partir
    de cadenas de dos bits.
    """
    # TODO
    pass


conteos_phi_plus = conteos_bell["Phi+"][1000]
frecuencias_individuales = # TODO
print(frecuencias_individuales)


Completa:

| Qubit | Frecuencia de 0 | Frecuencia de 1 |
|---|---:|---:|
| Qubit 0 |  |  |
| Qubit 1 |  |  |

**Responde:**

1. ¿Qué distribución tiene cada qubit individualmente?
2. ¿Qué distribución tiene el sistema conjunto?
3. ¿Cómo puede cada qubit parecer aleatorio y, al mismo tiempo, estar perfectamente correlacionado con el otro?

**Respuesta:**

Escribe aquí tu respuesta.


## 6. Parte E. Entrelazamiento frente a mezcla clásica

### Ejercicio 10. Comparación con una mezcla clásica

Simula una mezcla clásica que produzca:

- $|00\rangle$ el 50 % de las veces.
- $|11\rangle$ el 50 % de las veces.

Compárala con el histograma del estado entrelazado $|\Phi^+\rangle$.


In [ ]:
rng = np.random.default_rng(SEMILLA)

resultados = rng.choice(
    ["00", "11"],
    size=1000,
    p=[0.5, 0.5]
)

conteos_mezcla = # TODO: construir diccionario de conteos
conteos_entrelazado = # TODO

print("Mezcla clásica:", conteos_mezcla)
print("Estado entrelazado Phi+:", conteos_entrelazado)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_histogram(conteos_mezcla, ax=axes[0], title="Mezcla clásica")
plot_histogram(conteos_entrelazado, ax=axes[1], title="Estado entrelazado")
plt.tight_layout()
plt.show()


**Responde:**

1. ¿Son iguales o parecidos los histogramas en la base computacional?
2. ¿Significa eso que la mezcla clásica y el estado entrelazado son lo mismo?
3. ¿Qué información no se observa al medir solo en la base computacional?
4. ¿Qué papel tiene la fase relativa en el estado entrelazado?

**Respuesta:**

Escribe aquí tu respuesta.


### Ejercicio 14. Preguntas finales

Responde de forma razonada:

1. ¿Por qué no toda superposición es entrelazamiento?
2. ¿Por qué $|+\rangle \otimes |+\rangle$ es separable?
3. ¿Por qué $\frac{|00\rangle+|11\rangle}{\sqrt{2}}$ es entrelazado?
4. ¿Por qué una mezcla clásica puede producir el mismo histograma que un estado entrelazado?
5. ¿Por qué hace falta medir en otras bases para distinguir ciertos estados?
6. ¿Por qué el entrelazamiento no permite comunicación instantánea?

**Respuesta:**

Escribe aquí tu respuesta.
